In [1]:
%pip install bm25s

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
# vanilla BM25s

import json, re
from pathlib import Path
import bm25s

DATA_PATH = Path(r"c:\datasources\ai\articles_df.json")

def strip_html(s: str) -> str:
    if not s:
        return ""
    s = re.sub(r"<[^>]+>", " ", s)
    s = s.replace("&nbsp;", " ").replace("&amp;", "&").replace("&quot;", '"').replace("&#39;", "'")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# 1) Load data
with DATA_PATH.open("r", encoding="utf-8") as f:
    rows = json.load(f)

# 2) Build corpus text (no stemming, no tricks)
corpus_texts = []
for r in rows:
    title = (r.get("Title") or "").strip()
    excerpt = strip_html(r.get("Excerpt") or "")
    corpus_texts.append(f"{title}\n{excerpt}".strip())

# 3) Tokenize WITHOUT stemming
corpus_tokens = bm25s.tokenize(
    corpus_texts,
    stopwords="en"   # stopwords only; no stemmer argument
)

# 4) Build BM25 index
retriever = bm25s.BM25()
retriever.index(corpus_tokens)

def bm25_search(query: str, k: int = 10):
    query_tokens = bm25s.tokenize(
        query,
        stopwords="en"   # again: no stemmer
    )

    doc_ids, scores = retriever.retrieve(query_tokens, k=k)
    doc_ids = doc_ids[0]
    scores = scores[0]

    results = []
    for rank, (idx, score) in enumerate(zip(doc_ids.tolist(), scores.tolist()), start=1):
        r = rows[idx]
        results.append({
            "rank": rank,
            "score": float(score),
            "ID": r.get("ID"),
            "Title": r.get("Title"),
            "Date": r.get("Date"),
            "Excerpt": strip_html(r.get("Excerpt") or "")[:500]
        })
    return results

# 5) Test
for r in bm25_search("tell me about death squads in Haiti", k=10):
    print(
        f'#{r["rank"]} score={r["score"]:.3f}  ID={r["ID"]}  Date={r["Date"]}\n'
        f'{r["Title"]}\n{r["Excerpt"]}\n'
    )

resource module not available on Windows


Split strings:   0%|          | 0/13851 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/13851 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/13851 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

#1 score=10.492  ID=13149  Date=2024-03-28
Haitian Community Defenders Fight US-Armed Death Squads and Puppet Governments
The Haitian Bald Headed Party (PHTK) has tyrannically ruled Haiti since 2011. The U.S. State Department, who unilaterally picked Ariel Henry to be Haiti’s prime minister in July of 2021, has now decided Henry no longer fits their interests and has forced him to step down. Henry was prevented from returning to Haiti on March 5 by paramilitary gangs who attempted to take the Toussaint Louverture International Airport, opening fire and hitting a plane bound for Cuba. The imperial forces responsible

#2 score=9.485  ID=13961  Date=2025-06-06
Erik Prince brings his mercenaries to Haiti. What could go wrong?
Haiti could be Erik Prince’s deadliest gambit yet. Prince's Blackwater reigned during the Global War on Terror, but left a legacy of disastrous mishaps , most infamously the 2007 Nisour massacre in Iraq, where Blackwater mercenaries killed 17 civilians . This, plus hi

In [1]:
%pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   ------------------------------------- -- 9.7/10.3 MB 46.1 MB/s eta 0:00:01
   ---------------------------------------- 10.3/10.3 MB 43.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/553.3 kB ? eta -:--:--
   --------------------------------------- 553.3/553.3 kB 34.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 2.9/2.9 MB 49.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 55.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/113.8 MB ? eta -:--:--
   ---- ----------------------------------- 13.9/113.8 MB 67.9 MB/s eta 0:00:02
   ---------- ----------------------------- 28.8/113.8 MB 71.2 MB/s eta 0:00:02
   --------------- --

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import json, re
from pathlib import Path
import numpy as np
import bm25s
import time

from sentence_transformers import SentenceTransformer

DATA_PATH = Path(r"c:\datasources\ai\articles_df.json")

def strip_html(s: str) -> str:
    if not s:
        return ""
    s = re.sub(r"<[^>]+>", " ", s)
    s = s.replace("&nbsp;", " ").replace("&amp;", "&").replace("&quot;", '"').replace("&#39;", "'")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# 1) Load dataset
with DATA_PATH.open("r", encoding="utf-8") as f:
    rows = json.load(f)

# 2) Build corpus (title-weighting optional; uncomment if you want)
corpus_texts = []
for r in rows:
    title = (r.get("Title") or "").strip()
    excerpt = strip_html(r.get("Excerpt") or "")
    # simple baseline:
    corpus_texts.append(f"{title}\n{excerpt}".strip())
    # optional title boost:
    # corpus_texts.append(f"{title}\n{title}\n{excerpt}".strip())

# 3) BM25 tokenize + index (no stemming)
corpus_tokens = bm25s.tokenize(corpus_texts, stopwords="en")
retriever = bm25s.BM25()
retriever.index(corpus_tokens)

# 4) Load embedding model (BGE small)
# Commonly used English small model:
BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"
embedder = SentenceTransformer(BGE_MODEL_NAME)

def _cosine_sim_matrix(q_vec: np.ndarray, doc_vecs: np.ndarray) -> np.ndarray:
    """
    q_vec: (d,) or (1,d)
    doc_vecs: (n,d)
    Returns (n,) cosine similarities.
    Assumes vectors are already L2-normalized.
    """
    if q_vec.ndim == 1:
        q_vec = q_vec[None, :]
    return (doc_vecs @ q_vec.T).reshape(-1)

def hybrid_bm25_bge_search(query: str, bm25_k: int = 50, final_k: int = 10, verbose: bool = True):
    t0 = time.perf_counter()

    # --- Stage 1: BM25 ---
    t_bm25_start = time.perf_counter()

    q_tokens = bm25s.tokenize(query, stopwords="en")
    doc_ids, bm25_scores = retriever.retrieve(q_tokens, k=bm25_k)

    cand_idxs = doc_ids[0].tolist()
    cand_bm25 = bm25_scores[0].tolist()

    t_bm25_end = time.perf_counter()

    # --- Stage 2: BGE rerank ---
    t_bge_start = time.perf_counter()

    q_text = f"query: {query}"
    cand_texts = [f"passage: {corpus_texts[i]}" for i in cand_idxs]

    q_emb = embedder.encode([q_text], normalize_embeddings=True)
    d_embs = embedder.encode(cand_texts, normalize_embeddings=True)

    sims = (d_embs @ q_emb[0])   # cosine since normalized
    rerank_order = np.argsort(-sims)

    t_bge_end = time.perf_counter()
    t1 = time.perf_counter()

    if verbose:
        print(
            f"BM25 retrieval ({bm25_k} docs): {(t_bm25_end - t_bm25_start)*1000:.1f} ms\n"
            f"BGE rerank ({bm25_k} → {final_k}): {(t_bge_end - t_bge_start)*1000:.1f} ms\n"
            f"Total query time: {(t1 - t0)*1000:.1f} ms"
        )

    # --- Build final results ---
    results = []
    for rank, j in enumerate(rerank_order[:final_k], start=1):
        idx = cand_idxs[j]
        r = rows[idx]
        results.append({
            "rank": rank,
            "bge_score": float(sims[j]),
            "bm25_score": float(cand_bm25[j]),
            "ID": r.get("ID"),
            "Title": r.get("Title"),
            "Date": r.get("Date"),
            "Excerpt": strip_html(r.get("Excerpt") or "")[:500],
        })

    return results

# ---- try it ----
results = hybrid_bm25_bge_search("tell me about death squads in Haiti", bm25_k=50, final_k=10)
for r in results:
    print(f'#{r["rank"]}  bge={r["bge_score"]:.3f}  bm25={r["bm25_score"]:.3f}  ID={r["ID"]}  Date={r["Date"]}')
    print(r["Title"])
    print(r["Excerpt"])
    print()


resource module not available on Windows


Split strings:   0%|          | 0/13851 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/13851 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/13851 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25 retrieval (50 docs): 13.1 ms
BGE rerank (50 → 10): 4429.6 ms
Total query time: 4442.7 ms
#1  bge=0.833  bm25=9.485  ID=13961  Date=2025-06-06
Erik Prince brings his mercenaries to Haiti. What could go wrong?
Haiti could be Erik Prince’s deadliest gambit yet. Prince's Blackwater reigned during the Global War on Terror, but left a legacy of disastrous mishaps , most infamously the 2007 Nisour massacre in Iraq, where Blackwater mercenaries killed 17 civilians . This, plus his willingness in recent years to work for foreign governments in conflicts and for law enforcement across the globe, have made Prince one of the world’s most controversial entrepreneurs. A desperate Haiti has now hired him to “ cond

#2  bge=0.827  bm25=10.492  ID=13149  Date=2024-03-28
Haitian Community Defenders Fight US-Armed Death Squads and Puppet Governments
The Haitian Bald Headed Party (PHTK) has tyrannically ruled Haiti since 2011. The U.S. State Department, who unilaterally picked Ariel Henry to be Haiti